# ML Testing & MLOps — Building a Model Promotion Gate

**Dataset:** `sklearn.datasets.fetch_california_housing` — 20,640 real 1990 California census
block-group records (median income, house age, rooms, population, occupancy, latitude/longitude),
with the target being median house value. Fetched at build time via scikit-learn's built-in loader
(downloads from the StatLib archive the first time it runs).

## Problem statement

Your team has trained a **challenger** regression model that a data scientist wants to promote to
replace the **champion** model currently serving housing-price predictions. The data scientist's
evidence for promoting it is: "the challenger's test-set MAE is lower than the champion's." As the
QA engineer on this team, you know that a single point-estimate comparison on one test split is not
a promotion decision — it can't tell you whether the improvement is real, whether it holds across
sub-populations, or whether the challenger is even stable across re-runs.

Your job is to build a **promotion gate**: an automated test suite that a champion-vs-challenger
comparison must pass before the challenger is allowed to ship. Concretely, you will produce:

1. A **paired bootstrap confidence interval** on the difference in MAE between champion and
   challenger (not just the point estimate) — to decide if the improvement is statistically
   distinguishable from noise.
2. A **paired non-parametric significance test** (Wilcoxon signed-rank on per-sample absolute
   errors) as a second, assumption-light check on the same question.
3. A **segment regression check**: split the test set into income-level segments (quartiles of
   `MedInc`) and verify the challenger does not get *meaningfully worse* than the champion on any
   single segment, even if it wins on average — an aggregate win can hide a regression for a
   sub-population.
4. A **seed-sensitivity check**: retrain the challenger across several random seeds and verify its
   test MAE doesn't swing more than a tolerance — a model whose quality depends heavily on random
   luck is not something you can safely promote, because the model you validated may not be the
   model you ship.
5. A single `promotion_gate(...)` function that runs all of the above and returns a structured
   pass/fail report with a clear reason for any failure.

By the end, you should be able to run `promotion_gate(...)` on champion vs. challenger predictions
and get back something you could paste into a CI log or a model-release checklist.


## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from scipy import stats

RANDOM_STATE = 42


## Load the data and define the champion / challenger split

We hold out a single test set that both models will be scored on (this is deliberate: a promotion
gate compares two models on the *same* data, never on separately-sampled test sets).

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.rename(columns={"MedHouseVal": "target"})

FEATURES = [c for c in df.columns if c != "target"]
X = df[FEATURES]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

print(df.shape)
df.head()


## Your task

Fill in the TODOs below. Function signatures and return shapes are given — keep them as-is, the
solution cells and the final `promotion_gate` call assume these names and shapes.

In [ ]:
def train_champion(X_train, y_train):
    """The current production model: plain Ridge regression. Return the fitted estimator."""
    # TODO
    raise NotImplementedError


def train_challenger(X_train, y_train, random_state=RANDOM_STATE):
    """The candidate replacement: a RandomForestRegressor (n_estimators=200).
    Return the fitted estimator."""
    # TODO
    raise NotImplementedError


In [ ]:
def paired_bootstrap_mae_diff_ci(y_true, pred_champion, pred_challenger,
                                  n_boot=2000, ci=0.95, random_state=RANDOM_STATE):
    """
    Resample the *indices* of the test set (with replacement) n_boot times. For each
    resample, compute MAE(champion) - MAE(challenger) on that resampled subset using the
    SAME indices for both models (paired resampling, not two independent resamples).

    Return (point_estimate, lower, upper) for the `ci` central interval of that difference,
    where point_estimate is the difference computed on the full (non-resampled) test set.
    A positive value means the challenger has lower error (champion_err - challenger_err > 0).
    """
    # TODO
    raise NotImplementedError


def paired_wilcoxon_test(y_true, pred_champion, pred_challenger):
    """
    Compute per-sample absolute errors for both models, then run a paired Wilcoxon
    signed-rank test (scipy.stats.wilcoxon) on (abs_err_champion - abs_err_challenger).

    Return (statistic, p_value).
    """
    # TODO
    raise NotImplementedError


In [ ]:
def segment_regression_check(y_true, pred_champion, pred_challenger, segment_labels,
                              max_allowed_regression=0.03):
    """
    segment_labels: an array/Series aligned with y_true, one label per test row (we'll pass
    income quartiles: 'Q1'..'Q4').

    For each distinct segment, compute MAE(champion) and MAE(challenger) on just that
    segment. The challenger REGRESSES on a segment if its MAE is more than
    `max_allowed_regression` (relative, e.g. 0.03 = 3%) worse than the champion's MAE on
    that same segment, i.e. (challenger_mae - champion_mae) / champion_mae > max_allowed_regression.

    Return a dict: {segment_label: {"champion_mae": ..., "challenger_mae": ...,
    "relative_change": ..., "regressed": bool}, ...}
    """
    # TODO
    raise NotImplementedError


def seed_sensitivity_check(X_train, y_train, X_test, y_test, seeds, max_std=0.01):
    """
    Retrain the CHALLENGER once per seed in `seeds` (reuse train_challenger), score each
    retrained model's MAE on X_test/y_test, and compute the standard deviation of those
    MAE values.

    Return (mae_values: list[float], std: float, passed: bool) where passed = std <= max_std.
    """
    # TODO
    raise NotImplementedError


In [ ]:
def promotion_gate(X_train, y_train, X_test, y_test, segment_labels,
                    bootstrap_ci_level=0.95, max_allowed_regression=0.03,
                    seeds=(0, 1, 2, 3, 4), max_seed_std=0.01):
    """
    Run the full promotion gate and return a report dict with at least these keys:
      - "champion_mae", "challenger_mae": floats on the full test set
      - "bootstrap_ci": (point_estimate, lower, upper) from paired_bootstrap_mae_diff_ci
      - "wilcoxon": (statistic, p_value) from paired_wilcoxon_test
      - "segment_report": the dict from segment_regression_check
      - "seed_sensitivity": the tuple from seed_sensitivity_check
      - "passed": bool — True only if ALL of:
            * bootstrap CI lower bound > 0 (challenger reliably better, not just on average)
            * wilcoxon p_value < 0.05
            * no segment has "regressed" == True
            * seed sensitivity "passed" == True
      - "reasons": list[str] of every gate that failed (empty list if passed)
    """
    # TODO
    raise NotImplementedError


In [ ]:
# Build income-quartile segment labels for the test set, then run the gate.
# TODO: compute `income_quartiles` as a pandas Series of labels ('Q1'..'Q4') aligned with
# X_test, using pd.qcut on X_test["MedInc"].

income_quartiles = None  # TODO

report = None  # TODO: call promotion_gate(...) and assign the result
report


### Write-up

TODO: in 3–5 sentences — did the challenger pass the gate? If it failed, which specific check
failed it, and why might that check matter more in production than the raw aggregate MAE
improvement? If it passed, which check came closest to failing, and what would you watch in
production monitoring to catch that check degrading over time after the challenger ships?

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def train_champion(X_train, y_train):
    model = Ridge(random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    return model


def train_challenger(X_train, y_train, random_state=RANDOM_STATE):
    model = RandomForestRegressor(n_estimators=200, random_state=random_state, n_jobs=-1)
    model.fit(X_train, y_train)
    return model


def paired_bootstrap_mae_diff_ci(y_true, pred_champion, pred_challenger,
                                  n_boot=2000, ci=0.95, random_state=RANDOM_STATE):
    y_true = np.asarray(y_true)
    pred_champion = np.asarray(pred_champion)
    pred_challenger = np.asarray(pred_challenger)
    n = len(y_true)
    rng = np.random.default_rng(random_state)

    point_estimate = (
        mean_absolute_error(y_true, pred_champion)
        - mean_absolute_error(y_true, pred_challenger)
    )

    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        champion_mae = mean_absolute_error(y_true[idx], pred_champion[idx])
        challenger_mae = mean_absolute_error(y_true[idx], pred_challenger[idx])
        diffs[b] = champion_mae - challenger_mae

    alpha = 1 - ci
    lower, upper = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    return point_estimate, lower, upper


def paired_wilcoxon_test(y_true, pred_champion, pred_challenger):
    y_true = np.asarray(y_true)
    abs_err_champion = np.abs(y_true - np.asarray(pred_champion))
    abs_err_challenger = np.abs(y_true - np.asarray(pred_challenger))
    statistic, p_value = stats.wilcoxon(abs_err_champion - abs_err_challenger)
    return statistic, p_value


def segment_regression_check(y_true, pred_champion, pred_challenger, segment_labels,
                              max_allowed_regression=0.03):
    y_true = np.asarray(y_true)
    pred_champion = np.asarray(pred_champion)
    pred_challenger = np.asarray(pred_challenger)
    segment_labels = np.asarray(segment_labels)

    report = {}
    for seg in sorted(pd.unique(segment_labels)):
        mask = segment_labels == seg
        champion_mae = mean_absolute_error(y_true[mask], pred_champion[mask])
        challenger_mae = mean_absolute_error(y_true[mask], pred_challenger[mask])
        relative_change = (challenger_mae - champion_mae) / champion_mae
        report[seg] = {
            "champion_mae": champion_mae,
            "challenger_mae": challenger_mae,
            "relative_change": relative_change,
            "regressed": relative_change > max_allowed_regression,
        }
    return report


def seed_sensitivity_check(X_train, y_train, X_test, y_test, seeds, max_std=0.01):
    mae_values = []
    for seed in seeds:
        model = train_challenger(X_train, y_train, random_state=seed)
        preds = model.predict(X_test)
        mae_values.append(mean_absolute_error(y_test, preds))
    std = float(np.std(mae_values))
    return mae_values, std, std <= max_std


def promotion_gate(X_train, y_train, X_test, y_test, segment_labels,
                    bootstrap_ci_level=0.95, max_allowed_regression=0.03,
                    seeds=(0, 1, 2, 3, 4), max_seed_std=0.01):
    champion = train_champion(X_train, y_train)
    challenger = train_challenger(X_train, y_train)

    pred_champion = champion.predict(X_test)
    pred_challenger = challenger.predict(X_test)

    champion_mae = mean_absolute_error(y_test, pred_champion)
    challenger_mae = mean_absolute_error(y_test, pred_challenger)

    bootstrap_ci = paired_bootstrap_mae_diff_ci(
        y_test, pred_champion, pred_challenger, ci=bootstrap_ci_level
    )
    wilcoxon = paired_wilcoxon_test(y_test, pred_champion, pred_challenger)
    segment_report = segment_regression_check(
        y_test, pred_champion, pred_challenger, segment_labels,
        max_allowed_regression=max_allowed_regression,
    )
    seed_sensitivity = seed_sensitivity_check(
        X_train, y_train, X_test, y_test, seeds, max_std=max_seed_std
    )

    reasons = []
    if not (bootstrap_ci[1] > 0):
        reasons.append("bootstrap CI for MAE improvement includes zero or favors champion")
    if not (wilcoxon[1] < 0.05):
        reasons.append("Wilcoxon signed-rank test not significant at p<0.05")
    regressed_segments = [seg for seg, r in segment_report.items() if r["regressed"]]
    if regressed_segments:
        reasons.append(f"regression on segment(s): {regressed_segments}")
    if not seed_sensitivity[2]:
        reasons.append(f"challenger MAE std across seeds ({seed_sensitivity[1]:.4f}) exceeds {max_seed_std}")

    return {
        "champion_mae": champion_mae,
        "challenger_mae": challenger_mae,
        "bootstrap_ci": bootstrap_ci,
        "wilcoxon": wilcoxon,
        "segment_report": segment_report,
        "seed_sensitivity": seed_sensitivity,
        "passed": len(reasons) == 0,
        "reasons": reasons,
    }


income_quartiles = pd.qcut(X_test["MedInc"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
report = promotion_gate(X_train, y_train, X_test, y_test, income_quartiles)
report
```

**Why this is the right shape for a promotion gate:** a champion/challenger comparison that stops
at "the mean MAE went down" is exactly the kind of check an SQA background should be suspicious of —
it's one sample statistic from one split, with no notion of variance. The bootstrap CI and Wilcoxon
test both ask "is this difference distinguishable from noise?" from two different angles (parametric
resampling vs. a distribution-free rank test) — if they disagree, that's itself a signal to
investigate rather than ship. The segment check is a fairness/regression-hiding guard: a model can
win in aggregate by getting much better on the largest segment while quietly getting worse on a
smaller one, which `max_allowed_regression` catches on a per-segment basis instead of averaging it
away. The seed-sensitivity check exists because the model you evaluate in a notebook is not
necessarily the model that gets retrained and deployed by a pipeline — if a model's quality varies a
lot with its random seed, the promotion decision you made may not describe the model you actually
ship next week.

</details>